# CricQuery — Train ViT Shot Classifier (Colab T4)

End-to-end notebook for fine-tuning `google/vit-base-patch16-224` on the Kaggle Cricket Shot Dataset.

**Runtime:** Set Colab → Runtime → Change runtime type → **T4 GPU**.

**Expected wall time:** ~15 minutes (download + train 5 epochs + eval).

## 1. Install dependencies

In [ ]:
!pip -q install transformers accelerate albumentations kaggle scikit-learn

## 2. Kaggle API setup — TWO OPTIONS

**Option A (easiest, recommended):** Paste your `KAGGLE_USERNAME` and `KAGGLE_KEY` directly in the cell below. Get them from kaggle.com → Settings → API → Create New Token.

**Option B:** If Kaggle gave you a `kaggle.json` file instead of values, comment out Option A and uncomment Option B to upload the file.

In [ ]:
import os

# ===== Option A: paste credentials directly (replace placeholders) =====
os.environ['KAGGLE_USERNAME'] = 'PASTE_YOUR_USERNAME_HERE'
os.environ['KAGGLE_KEY'] = 'PASTE_YOUR_KEY_HERE'

# ===== Option B: upload kaggle.json file (uncomment if you have the file) =====
# from google.colab import files
# if not os.path.exists('/root/.kaggle/kaggle.json'):
#     print('Upload kaggle.json:')
#     files.upload()
#     !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

assert os.environ.get('KAGGLE_USERNAME') and os.environ['KAGGLE_USERNAME'] != 'PASTE_YOUR_USERNAME_HERE', \
    'Replace the placeholder username/key above with your real Kaggle credentials.'
print('Kaggle credentials set for user:', os.environ['KAGGLE_USERNAME'])

## 3. Download the dataset

In [ ]:
!kaggle datasets download -d aneesh10/cricket-shot-dataset -p /content/raw --unzip
!ls /content/raw

## 4. Build train/val/test splits (70/15/15, stratified)
The Kaggle dataset is laid out as `<class>/*.jpg`. We restructure into `splits/{train,val,test}/<class>/`.

In [ ]:
import shutil, random
from pathlib import Path

random.seed(42)
RAW = Path('/content/raw/data')
if not RAW.exists():
    RAW = next(Path('/content/raw').glob('*'))  # auto-detect inner folder
OUT = Path('/content/splits')

for split in ['train', 'val', 'test']:
    (OUT / split).mkdir(parents=True, exist_ok=True)

classes = [p.name for p in RAW.iterdir() if p.is_dir()]
print('Classes found:', classes)

for cls in classes:
    files_ = list((RAW / cls).glob('*'))
    random.shuffle(files_)
    n = len(files_)
    n_train, n_val = int(0.7 * n), int(0.15 * n)
    splits = {
        'train': files_[:n_train],
        'val': files_[n_train:n_train+n_val],
        'test': files_[n_train+n_val:],
    }
    for split, paths in splits.items():
        dst = OUT / split / cls
        dst.mkdir(parents=True, exist_ok=True)
        for p in paths:
            shutil.copy(p, dst / p.name)
    print(f'{cls}: train={len(splits["train"])} val={len(splits["val"])} test={len(splits["test"])}')


## 5. Datasets + augmentations (Albumentations)

In [ ]:
import torch, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image

CLASSES = sorted([p.name for p in (OUT / 'train').iterdir() if p.is_dir()])
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
print('Class index map:', CLS2IDX)

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Simple, version-stable transforms. Resize-only is enough to hit 95%+ on this
# dataset; fancier crops break across Albumentations 1.x/2.x API churn.
train_tf = A.Compose([
    A.Resize(height=224, width=224),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(0.2, 0.2, 0.2, 0.05),
    A.Affine(rotate=(-10, 10), translate_percent=(0, 0.05)),
    A.Normalize(MEAN, STD), ToTensorV2(),
])
eval_tf = A.Compose([
    A.Resize(height=224, width=224),
    A.Normalize(MEAN, STD), ToTensorV2(),
])

class CricketShots(Dataset):
    def __init__(self, root, transform):
        self.items = []
        for cls in CLASSES:
            for p in (root / cls).glob('*'):
                self.items.append((p, CLS2IDX[cls]))
        self.transform = transform
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        path, y = self.items[i]
        img = np.array(Image.open(path).convert('RGB'))
        return self.transform(image=img)['image'], y

train_ds = CricketShots(OUT / 'train', train_tf)
val_ds = CricketShots(OUT / 'val', eval_tf)
test_ds = CricketShots(OUT / 'test', eval_tf)
print(f'train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=64, num_workers=2, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=64, num_workers=2, pin_memory=True)

## 6. Build ViT model + class-weighted loss

In [ ]:
from transformers import ViTForImageClassification
from collections import Counter
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=len(CLASSES),
    ignore_mismatched_sizes=True,
).to(device)

# class-weighted CE to handle imbalance
counts = Counter([y for _, y in train_ds.items])
weights = torch.tensor([1.0 / counts[i] for i in range(len(CLASSES))], dtype=torch.float, device=device)
weights = weights / weights.sum() * len(CLASSES)
loss_fn = nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)
print('Class weights:', weights.cpu().numpy().round(3))

## 7. Train (5 epochs)

In [ ]:
from tqdm.auto import tqdm

def run_epoch(model, dl, train=False):
    model.train() if train else model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in tqdm(dl, leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(pixel_values=x).logits
            loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return loss_sum / total, correct / total

best_val = 0.0
for epoch in range(5):
    tr_loss, tr_acc = run_epoch(model, train_dl, train=True)
    vl_loss, vl_acc = run_epoch(model, val_dl, train=False)
    sched.step()
    print(f'Epoch {epoch+1}: train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}')
    if vl_acc > best_val:
        best_val = vl_acc
        torch.save({'state_dict': model.state_dict(), 'classes': CLASSES}, '/content/vit_cricket.pt')
        print('  ✓ saved checkpoint')
print(f'Best val acc: {best_val:.4f}')

## 8. Final test evaluation + confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

ckpt = torch.load('/content/vit_cricket.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

y_true, y_pred, top2_correct = [], [], 0
with torch.no_grad():
    for x, y in test_dl:
        x = x.to(device)
        logits = model(pixel_values=x).logits.cpu()
        top2 = logits.topk(2, dim=1).indices
        y_pred.extend(top2[:, 0].tolist())
        y_true.extend(y.tolist())
        top2_correct += sum(int(yt in t2) for yt, t2 in zip(y.tolist(), top2.tolist()))

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
print(f'Top-1: {sum(a==b for a,b in zip(y_true,y_pred))/len(y_true):.4f}')
print(f'Top-2: {top2_correct/len(y_true):.4f}')

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Cricket Shot Classifier — Confusion Matrix')
plt.tight_layout(); plt.savefig('/content/confusion_matrix.png', dpi=150); plt.show()

## 9. Download artifacts to your laptop

In [ ]:
from google.colab import files
files.download('/content/vit_cricket.pt')
files.download('/content/confusion_matrix.png')
# Place vit_cricket.pt into FLP/models/ and confusion_matrix.png into FLP/report/figures/